# Week 80, Day 1: Project: a full API (design)

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w080d1.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
STATUSES = ("open", "in_progress", "waiting", "resolved", "closed")
TRANSITIONS = {
    "open": {"in_progress": {"agent", "admin"}, "closed": {"customer", "admin"}},
    "in_progress": {"waiting": {"agent", "admin"}, "resolved": {"agent", "admin"}},
    "waiting": {"in_progress": {"customer", "agent", "admin"}},
    "resolved": {"closed": {"customer", "admin"}, "in_progress": {"customer", "agent", "admin"}},
    "closed": {},
}
for role in ("customer", "agent"):
    moves = [(old, new) for old in STATUSES for new in TRANSITIONS[old] if role in TRANSITIONS[old][new]]
    print(role, moves)

> **Key idea.** Design turns a brief into testable decisions: resources and routes, validation rules, a state machine for the resource's life, a permission table, and one error format. Keep them in a domain module with no web or database code, so the rules can be tested alone and shared by every layer. Build in layers: web on top of domain and database.

### Exercise: Step 1: resources and routes

Using the `ROUTES` table (given), write `docs_table(routes)`: the routes as a Markdown table, starting with the lines `| Method | Path | What it does | Success |` and `|---|---|---|---|`, then one line per route: `| METHOD | \`PATH\` | WHAT | CODE PHRASE |`, with the status's phrase from `HTTPStatus`; lines joined with newlines. Then `match_route(routes, method, path)`, which decides what a request would hit: split the path and each route's template into segments at `/` (ignoring the slashes at either end); a template segment in braces, such as `{ticket_id}`, matches a segment made only of digits and captures it as an `int`; other segments must be equal. Return `(200, (template, params))` for the first route that matches both path and method, `(405, methods)` (a sorted list of the methods whose routes match the path) if the path matches but the method doesn't, or `(404, None)`.

In [ ]:
from http import HTTPStatus

ROUTES = [
    ("POST", "/users", "sign up as a customer", 201),
    ("POST", "/tokens", "log in and get a token", 201),
    ("GET", "/me", "who am I", 200),
    ("GET", "/tickets", "list tickets (customers see their own)", 200),
    ("POST", "/tickets", "open a ticket", 201),
    ("GET", "/tickets/{ticket_id}", "read a ticket", 200),
    ("PATCH", "/tickets/{ticket_id}", "change a ticket's title, body or priority", 200),
    ("DELETE", "/tickets/{ticket_id}", "delete a ticket (admins)", 204),
    ("POST", "/tickets/{ticket_id}/transitions", "change a ticket's status", 200),
    ("PUT", "/tickets/{ticket_id}/assignee", "assign a ticket to an agent", 200),
    ("GET", "/tickets/{ticket_id}/comments", "list a ticket's comments", 200),
    ("POST", "/tickets/{ticket_id}/comments", "add a comment or internal note", 201),
    ("GET", "/stats", "ticket counts (staff)", 200),
    ("GET", "/healthz", "is the process alive", 200),
    ("GET", "/readyz", "can it serve requests", 200),
]


def docs_table(routes):
    ___


def match_route(routes, method, path):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_docs():\n    \"a Markdown reference\"\n    lines = docs_table(ROUTES).split(\"\\n\")\n    assert lines[:3] == [\"| Method | Path | What it does | Success |\", \"|---|---|---|---|\", \"| POST | `/users` | sign up as a customer | 201 Created |\"]\n    assert len(lines) == len(ROUTES) + 2 and \"| DELETE | `/tickets/{ticket_id}` | delete a ticket (admins) | 204 No Content |\" in lines\n\ndef test_match():\n    \"routing decisions\"\n    assert match_route(ROUTES, \"GET\", \"/tickets/7\") == (200, (\"/tickets/{ticket_id}\", {\"ticket_id\": 7}))\n    assert match_route(ROUTES, \"POST\", \"/tickets/12/comments\") == (200, (\"/tickets/{ticket_id}/comments\", {\"ticket_id\": 12}))\n    assert match_route(ROUTES, \"GET\", \"/tickets/\") == (200, (\"/tickets\", {}))\n    assert match_route(ROUTES, \"DELETE\", \"/tickets\") == (405, [\"GET\", \"POST\"])\n    assert match_route(ROUTES, \"PATCH\", \"/tickets/3/comments\") == (405, [\"GET\", \"POST\"])\n    assert match_route(ROUTES, \"GET\", \"/tickets/seven\") == (404, None)\n    assert match_route(ROUTES, \"GET\", \"/tickets/7/history\") == (404, None) and match_route(ROUTES, \"GET\", \"/\") == (404, None)")

### Exercise: Step 2: what a ticket must be

Write `validate_ticket(data, partial=False)` with the given `ValidationError`. If `data` isn't a dict, raise `ValidationError({"_": "expected a JSON object"})`. Collect errors in a dict: unknown fields under `"_"` as `"unexpected field(s): A, B"` (sorted); for `title` and `body`, a missing field is `"required"` (unless `partial`), and a value that isn't a string or whose stripped length is outside the limits is `"3 to 120 characters"` or `"1 to 5000 characters"`; a `priority` that isn't one of `PRIORITIES` is `"one of low, normal, high, urgent"`. With `partial` and an empty dict, the error under `"_"` is `"nothing to change"`. If there are errors, raise `ValidationError(errors)`; otherwise return the clean values: stripped strings, the priority, and (when not `partial`) `"priority": "normal"` if it was left out.

In [ ]:
PRIORITIES = ("low", "normal", "high", "urgent")

class ValidationError(Exception):
    """Raised with a dict of field -> message."""

    def __init__(self, errors):
        super().__init__("; ".join(f"{k}: {v}" for k, v in sorted(errors.items())))
        self.errors = errors


def validate_ticket(data, partial=False):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def _errors(data, partial=False):\n    try:\n        validate_ticket(data, partial)\n    except ValidationError as e:\n        return e.errors\n    raise AssertionError(f\"{data!r} should be refused\")\n\ndef test_valid():\n    \"clean values\"\n    assert validate_ticket({\"title\": \"  Printer on fire \", \"body\": \"Smoke everywhere\\n\"}) == {\"title\": \"Printer on fire\", \"body\": \"Smoke everywhere\", \"priority\": \"normal\"}\n    assert validate_ticket({\"title\": \"abc\", \"body\": \"x\", \"priority\": \"urgent\"})[\"priority\"] == \"urgent\"\n    assert validate_ticket({\"priority\": \"low\"}, partial=True) == {\"priority\": \"low\"}\n    assert validate_ticket({\"title\": \"x\" * 120, \"body\": \"y\" * 5000}, partial=True)[\"title\"] == \"x\" * 120\n\ndef test_invalid():\n    \"every problem at once\"\n    assert _errors({\"title\": \"hi\", \"priority\": \"asap\", \"status\": \"closed\", \"id\": 9}) == {\n        \"_\": \"unexpected field(s): id, status\", \"title\": \"3 to 120 characters\", \"body\": \"required\", \"priority\": \"one of low, normal, high, urgent\"}\n    assert _errors({\"title\": \"   ab   \", \"body\": \"   \"}) == {\"title\": \"3 to 120 characters\", \"body\": \"1 to 5000 characters\"}\n    assert _errors({\"title\": 123, \"body\": None}) == {\"title\": \"3 to 120 characters\", \"body\": \"1 to 5000 characters\"}\n    assert _errors({\"title\": \"x\" * 121, \"body\": \"y\" * 5001}) == {\"title\": \"3 to 120 characters\", \"body\": \"1 to 5000 characters\"}\n    assert _errors({}, partial=True) == {\"_\": \"nothing to change\"} and _errors([\"not\", \"a\", \"dict\"]) == {\"_\": \"expected a JSON object\"}\n    try:\n        validate_ticket({\"title\": \"x\"})\n    except ValidationError as e:\n        assert str(e) == \"body: required; title: 3 to 120 characters\"")

### Exercise: Step 3: a ticket's life

With the given `ROLES`, `STATUSES` and `TRANSITIONS`, write `can_transition(current, new, role)` (`True` if the table allows it; unknown statuses just give `False`) and `next_statuses(current, role)` (the statuses that role may move to from `current`, in `STATUSES` order). Then `check_machine(transitions)`, returning a sorted list of problems with any transition table: `"S: missing"` for a status in `STATUSES` with no entry; `"S: not a status"` for an entry that isn't one; for each move, `"S->N: not a status"`, `"S->N: goes nowhere"` (N is S), and `"S->N: bad roles"` (no roles, or a role not in `ROLES`); `"closed: must be final"` if `closed` has any moves; and `"S: unreachable from open"` for each status that can't be reached from `open` by following moves.

In [ ]:
ROLES = ("customer", "agent", "admin")
STATUSES = ("open", "in_progress", "waiting", "resolved", "closed")

TRANSITIONS = {
    "open": {"in_progress": {"agent", "admin"}, "closed": {"customer", "admin"}},
    "in_progress": {"waiting": {"agent", "admin"}, "resolved": {"agent", "admin"}},
    "waiting": {"in_progress": {"customer", "agent", "admin"}},
    "resolved": {"closed": {"customer", "admin"}, "in_progress": {"customer", "agent", "admin"}},
    "closed": {},
}


def can_transition(current, new, role):
    ___


def next_statuses(current, role):
    ___


def check_machine(transitions):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_moves():\n    \"the real machine\"\n    assert can_transition(\"open\", \"in_progress\", \"agent\") and not can_transition(\"open\", \"in_progress\", \"customer\")\n    assert can_transition(\"resolved\", \"in_progress\", \"customer\") and not can_transition(\"closed\", \"open\", \"admin\")\n    assert not can_transition(\"lost\", \"open\", \"admin\") and not can_transition(\"open\", \"lost\", \"admin\")\n    assert next_statuses(\"resolved\", \"customer\") == [\"in_progress\", \"closed\"] and next_statuses(\"open\", \"agent\") == [\"in_progress\"]\n    assert next_statuses(\"closed\", \"admin\") == [] and next_statuses(\"waiting\", \"customer\") == [\"in_progress\"]\n    assert check_machine(TRANSITIONS) == []\n\ndef test_check():\n    \"mistakes in a table\"\n    bad = {\"open\": {\"open\": {\"agent\"}, \"resolved\": {\"robot\"}, \"archived\": {\"admin\"}}, \"closed\": {\"open\": {\"admin\"}}, \"lost\": {}}\n    assert check_machine(bad) == [\"closed: must be final\", \"closed: unreachable from open\", \"in_progress: missing\",\n                                  \"in_progress: unreachable from open\", \"lost: not a status\", \"open->archived: not a status\",\n                                  \"open->open: goes nowhere\", \"open->resolved: bad roles\", \"resolved: missing\",\n                                  \"waiting: missing\", \"waiting: unreachable from open\"]\n    no_roles = {s: dict(m) for s, m in TRANSITIONS.items()}\n    no_roles[\"waiting\"] = {\"in_progress\": set()}\n    assert check_machine(no_roles) == [\"waiting->in_progress: bad roles\"]")

### Exercise: Step 4: who may do what

With the given `ROLES` and `PERMISSIONS`, write `allowed(role, action, is_owner=False)`: `True` if the role is in the action's "always" set, or `is_owner` is true and the role is in its "own" set. An action that isn't in `PERMISSIONS` raises `KeyError` (let the dict lookup do it). Then `permission_matrix(actions=None)`: a plain-text table of the actions (all of `PERMISSIONS` in order, or the list given) against `ROLES`. The first line is `action` padded to the width of the longest action name, two spaces, then each role padded to 8 characters with two spaces between; each other line is the action padded the same way, two spaces, then a cell per role padded to 8 with two spaces between: `yes` (always allowed), `own` (only with `is_owner`) or `-`. Strip trailing spaces from every line, and join the lines with newlines.

In [ ]:
ROLES = ("customer", "agent", "admin")

PERMISSIONS = {
    "create_ticket": ({"customer"}, set()),
    "view_ticket": ({"agent", "admin"}, {"customer"}),
    "list_all_tickets": ({"agent", "admin"}, set()),
    "edit_ticket": ({"admin"}, {"customer"}),
    "comment": ({"agent", "admin"}, {"customer"}),
    "internal_note": ({"agent", "admin"}, set()),
    "change_priority": ({"agent", "admin"}, set()),
    "assign": ({"agent", "admin"}, set()),
    "delete_ticket": ({"admin"}, set()),
    "view_stats": ({"agent", "admin"}, set()),
}


def allowed(role, action, is_owner=False):
    ___


def permission_matrix(actions=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_allowed():\n    \"always, own and never\"\n    assert allowed(\"customer\", \"create_ticket\") and not allowed(\"agent\", \"create_ticket\")\n    assert not allowed(\"customer\", \"view_ticket\") and allowed(\"customer\", \"view_ticket\", is_owner=True) and allowed(\"agent\", \"view_ticket\")\n    assert not allowed(\"customer\", \"internal_note\", is_owner=True) and allowed(\"admin\", \"delete_ticket\") and not allowed(\"agent\", \"delete_ticket\")\n    assert not allowed(\"intruder\", \"view_ticket\", is_owner=True)\n    try:\n        allowed(\"admin\", \"launch_rockets\")\n    except KeyError:\n        pass\n    else:\n        raise AssertionError(\"an unknown action must be an error\")\n\ndef test_matrix():\n    \"a table for the docs\"\n    lines = permission_matrix().split(\"\\n\")\n    assert lines[0] == \"action            customer  agent     admin\"\n    assert lines[2] == \"view_ticket       own       yes       yes\" and lines[-1] == \"view_stats        -         yes       yes\"\n    assert len(lines) == len(PERMISSIONS) + 1\n    assert permission_matrix([\"assign\", \"comment\"]) == \"action   customer  agent     admin\\nassign   -         yes       yes\\ncomment  own       yes       yes\"")

### Exercise: Step 5: one error format

Write `problem(status, detail, title=None, **extra)`: return `(body, status, headers)`, where `body` is `{"type": "about:blank", "title": TITLE, "status": status, "detail": detail}` plus any `extra` fields, TITLE is `title` or else the status's phrase from `HTTPStatus`, and `headers` is `{"Content-Type": "application/problem+json"}`. Then `validation_problem(error)` for a `ValidationError`: a `422` problem with detail `"N field(s) are invalid"` and an extra field `errors`: a list of `{"field": NAME, "message": MESSAGE}`, sorted by field name.

In [ ]:
from http import HTTPStatus

class ValidationError(Exception):
    """Raised with a dict of field -> message."""

    def __init__(self, errors):
        super().__init__("; ".join(f"{k}: {v}" for k, v in sorted(errors.items())))
        self.errors = errors


def problem(status, detail, title=None, **extra):
    ___


def validation_problem(error):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_problem():\n    \"RFC 9457 bodies\"\n    body, status, headers = problem(404, \"no ticket 7\")\n    assert body == {\"type\": \"about:blank\", \"title\": \"Not Found\", \"status\": 404, \"detail\": \"no ticket 7\"} and status == 404\n    assert headers == {\"Content-Type\": \"application/problem+json\"}\n    body, status, _ = problem(500, \"something went wrong\", incident=\"ab12cd34\")\n    assert body[\"incident\"] == \"ab12cd34\" and body[\"title\"] == \"Internal Server Error\"\n    assert problem(409, \"x\", title=\"Ticket changed\")[0][\"title\"] == \"Ticket changed\"\n\ndef test_validation():\n    \"a 422 listing the fields\"\n    body, status, headers = validation_problem(ValidationError({\"title\": \"3 to 120 characters\", \"body\": \"required\"}))\n    assert status == 422 and body[\"detail\"] == \"2 field(s) are invalid\" and headers[\"Content-Type\"] == \"application/problem+json\"\n    assert body[\"errors\"] == [{\"field\": \"body\", \"message\": \"required\"}, {\"field\": \"title\", \"message\": \"3 to 120 characters\"}]")

In [ ]:
from http import HTTPStatus

PRIORITIES = ("low", "normal", "high", "urgent")

STATUSES = ("open", "in_progress", "waiting", "resolved", "closed")

TRANSITIONS = {
    "open": {"in_progress": {"agent", "admin"}, "closed": {"customer", "admin"}},
    "in_progress": {"waiting": {"agent", "admin"}, "resolved": {"agent", "admin"}},
    "waiting": {"in_progress": {"customer", "agent", "admin"}},
    "resolved": {"closed": {"customer", "admin"}, "in_progress": {"customer", "agent", "admin"}},
    "closed": {},
}

class ValidationError(Exception):
    """Raised with a dict of field -> message."""

    def __init__(self, errors):
        super().__init__("; ".join(f"{k}: {v}" for k, v in sorted(errors.items())))
        self.errors = errors

def validate_ticket(data, partial=False):
    """Check a ticket body from a client. partial=True for PATCH: only the fields present are checked."""
    if not isinstance(data, dict):
        raise ValidationError({"_": "expected a JSON object"})
    errors, clean = {}, {}
    unknown = sorted(set(data) - {"title", "body", "priority"})
    if unknown:
        errors["_"] = f"unexpected field(s): {', '.join(unknown)}"
    for field in ("title", "body"):
        if field not in data:
            if not partial:
                errors[field] = "required"
            continue
        value = data[field]
        limit = (3, 120) if field == "title" else (1, 5000)
        if not isinstance(value, str) or not limit[0] <= len(value.strip()) <= limit[1]:
            errors[field] = f"{limit[0]} to {limit[1]} characters"
        else:
            clean[field] = value.strip()
    if "priority" in data:
        if data["priority"] not in PRIORITIES:
            errors["priority"] = f"one of {', '.join(PRIORITIES)}"
        else:
            clean["priority"] = data["priority"]
    elif not partial:
        clean["priority"] = "normal"
    if partial and not data:
        errors["_"] = "nothing to change"
    if errors:
        raise ValidationError(errors)
    return clean

def can_transition(current, new, role):
    """True if someone with this role may move a ticket from `current` to `new`."""
    return role in TRANSITIONS.get(current, {}).get(new, set())

def problem(status, detail, title=None, **extra):
    """An RFC 9457 problem-details response: (body, status, headers)."""
    body = {"type": "about:blank", "title": title or HTTPStatus(status).phrase, "status": status, "detail": detail}
    body.update(extra)
    return body, status, {"Content-Type": "application/problem+json"}

for data in ({"title": "Printer on fire", "body": "Smoke everywhere", "priority": "urgent"}, {"title": "hi", "status": "closed"}):
    try:
        print(201, validate_ticket(data))
    except ValidationError as e:
        body, status, headers = problem(422, "the request has invalid fields", errors=e.errors)
        print(status, headers["Content-Type"], body["errors"])

for old, new, role in (("open", "in_progress", "agent"), ("open", "resolved", "agent"), ("resolved", "closed", "customer")):
    print(f"{old} -> {new} by {role}:", "allowed" if can_transition(old, new, role) else problem(403, f"you can't move a ticket from {old} to {new}")[0]["detail"])

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

## The brief

A small company wants a **helpdesk API**: a back end for its support website and its staff's app. The brief, as the support manager wrote it:

> Customers sign up with their email, log in, and open **tickets** describing a problem (a title, a description, and how urgent it is). They can see and comment on their own tickets, but nobody else's. **Agents** see every ticket, pick them up, ask customers questions, add **internal notes** that customers never see, change priorities, and mark tickets resolved. Customers can close a resolved ticket, or reopen it if the fix didn't work. **Admins** can do everything agents can, and can also delete tickets and see everything. We want to know how many tickets are in each state, and how many nobody has picked up. It must be secure (it holds customers' details), it must not lose or double-count anything, and we need to be able to deploy updates without taking it down.

That's typical of a real brief: clear about what people want to do, vague about the details. Design turns it into decisions precise enough to build and test. Every decision today becomes a small piece of code in a module called `helpdesk_domain`, which knows nothing about HTTP or databases, so the rules can be tested on their own and shared by every other part.

## A plan for the week

The project follows the same order a team would, one layer a day, each building on the last:

| Day | Layer | Module | From |
|---|---|---|---|
| 1 | design: routes, rules, states, permissions, errors | `helpdesk_domain` | Weeks 74, 77 |
| 2 | database: schema, migrations, queries, transactions | `helpdesk_db` | Week 75 |
| 3 | endpoints and authentication | `helpdesk_app`, `helpdesk_security` | Weeks 77, 79 |
| 4 | tests and continuous integration | `tests/`, a CI workflow | Weeks 72, 74 |
| 5 | deployment: settings, health checks, logs, containers, migrations in production | | Weeks 77, 78 |
| 6 | review: the whole system, its weak points, and one more feature | | |

The shape inside is **layered**: the web layer (routes, requests, responses) calls the domain rules and the database layer; the domain layer calls nothing. That's the "functional core, imperative shell" idea from Week 36: the decisions live in plain functions that are easy to test, and the parts that talk to the outside world are thin.

```
HTTP request -> helpdesk_app (parse, authenticate, check permission) -> helpdesk_domain (rules)
                                                                    -> helpdesk_db (SQL) -> sqlite
```

Each step today gives you the earlier steps' code where you need it, so you can concentrate on the new piece. Tomorrow's page includes the finished `helpdesk_domain` module.

## Step 1: resources and routes

The nouns in the brief become **resources**: users, tokens (a login is creating a token), tickets, and each ticket's comments, assignee and status changes. Following Week 77's REST conventions, the routes are:

```python
ROUTES = [
    ("POST", "/users", "sign up as a customer", 201),
    ("POST", "/tokens", "log in and get a token", 201),
    ("GET", "/me", "who am I", 200),
    ("GET", "/tickets", "list tickets (customers see their own)", 200),
    ("POST", "/tickets", "open a ticket", 201),
    ("GET", "/tickets/{ticket_id}", "read a ticket", 200),
    ("PATCH", "/tickets/{ticket_id}", "change a ticket's title, body or priority", 200),
    ("DELETE", "/tickets/{ticket_id}", "delete a ticket (admins)", 204),
    ("POST", "/tickets/{ticket_id}/transitions", "change a ticket's status", 200),
    ("PUT", "/tickets/{ticket_id}/assignee", "assign a ticket to an agent", 200),
    ("GET", "/tickets/{ticket_id}/comments", "list a ticket's comments", 200),
    ("POST", "/tickets/{ticket_id}/comments", "add a comment or internal note", 201),
    ("GET", "/stats", "ticket counts (staff)", 200),
    ("GET", "/healthz", "is the process alive", 200),
    ("GET", "/readyz", "can it serve requests", 200),
]
```

Two design decisions are worth noticing. A status change is `POST /tickets/{ticket_id}/transitions`, a resource of its own, rather than a `PATCH` of the `status` field, because a transition has rules (who may make it, from which status) and a history worth recording. And assigning is a `PUT` of the ticket's `assignee`, because setting it twice to the same agent should have the same effect as once (`PUT` is idempotent).

A route table written as data does more than document the API: the same table produces the README's reference section, and a test can check that every route in it really exists and answers with the status code it promises (that test comes on Thursday).

## Step 2: what a ticket must be

A ticket from a client has a `title`, a `body` and a `priority`. The rules, written precisely:

- `title`: a string of **3 to 120** characters once surrounding spaces are stripped;
- `body`: a string of **1 to 5000** characters, also stripped;
- `priority`: one of `low`, `normal`, `high`, `urgent`; it may be left out when creating (it becomes `normal`);
- **nothing else**: an `id`, a `status` or a `customer_id` sent by a client must be refused, not quietly applied (mass assignment, Week 79);
- for an update (`PATCH`), only the fields present are checked, and an empty update is an error.

One function, `validate_ticket`, holds these rules and **collects every problem** into a dict of field to message, raised as a `ValidationError`; the web layer turns that into a `422` response. Because it's one function, the create endpoint, the update endpoint, an import script and the tests all agree on what a valid ticket is.

## Step 3: a ticket's life

A ticket moves through **statuses**, and not every move makes sense: a closed ticket stays closed; only staff mark a ticket resolved; a customer can close a ticket that was resolved, or reopen it if the fix didn't work. A **state machine** says exactly which moves exist and who may make each, as a table:

```python
TRANSITIONS = {
    "open": {"in_progress": {"agent", "admin"}, "closed": {"customer", "admin"}},
    "in_progress": {"waiting": {"agent", "admin"}, "resolved": {"agent", "admin"}},
    "waiting": {"in_progress": {"customer", "agent", "admin"}},
    "resolved": {"closed": {"customer", "admin"}, "in_progress": {"customer", "agent", "admin"}},
    "closed": {},
}
```

```
          agent          agent           agent
  open ----------> in_progress <------> waiting
   |                  |    ^ customer, agent (reopen)
   | customer         | agent
   v                  v    |
 closed <---------- resolved
          customer
```

Writing the machine as data means the web layer needs only one check (`can_transition(old, new, role)`), the documentation can be drawn from the same table, and the table itself can be **checked** for mistakes that are easy to make when someone edits it later: a status with no entry, a move to a status that doesn't exist, a move from a status to itself, an unknown role, a way out of `closed`, or a status that can never be reached from `open`.

## Step 4: who may do what

The brief's sentences about people become a **permission table**: for each action, the roles that may always do it, and the roles that may do it only on **their own** tickets:

```python
PERMISSIONS = {
    "create_ticket": ({"customer"}, set()),
    "view_ticket": ({"agent", "admin"}, {"customer"}),
    "list_all_tickets": ({"agent", "admin"}, set()),
    "edit_ticket": ({"admin"}, {"customer"}),
    "comment": ({"agent", "admin"}, {"customer"}),
    "internal_note": ({"agent", "admin"}, set()),
    "change_priority": ({"agent", "admin"}, set()),
    "assign": ({"agent", "admin"}, set()),
    "delete_ticket": ({"admin"}, set()),
    "view_stats": ({"agent", "admin"}, set()),
}
```

One function, `allowed(role, action, is_owner)`, answers every question, so the web layer never contains a scattered `if user.role == "admin" or ...`. Two habits from Week 79 apply: the table is an **allow-list** (an action or role it doesn't mention is denied), and an action that isn't in the table should be an error in the code, not a silent "no", so a typo in an action name is found at once.

Notice how one decision in the table shapes the API's behaviour: customers may view only their own tickets, and the API answers a customer asking for someone else's ticket with `404 Not Found`, not `403 Forbidden`, so it doesn't even confirm that the ticket exists.

## Step 5: one error format

Every error from the API uses one format, the **problem details** of RFC 9457 (Week 77): a JSON body with `type`, `title`, `status` and `detail`, the media type `application/problem+json`, and extra fields when they help, such as the list of invalid fields for a `422`. With one helper building every error, clients handle all of them with one piece of code, and nobody has to remember the format when adding an endpoint.

The status codes the helpdesk uses, decided now so every endpoint agrees: `400` (the request can't be read: not JSON, a bad query parameter), `401` (not logged in, or the token is bad or expired), `403` (logged in, but not allowed), `404` (no such thing, or not one you may see), `405` (the route exists, the method doesn't), `409` (it conflicts with the current state: a duplicate email, a ticket that changed meanwhile), `422` (readable but invalid), `500` (our fault, with an incident id and no details, Week 79), `503` (not ready).

## Checkpoint

Here are today's decisions working together: a request is routed, its body validated, and the answer built, including the errors a client would see.

The finished `helpdesk_domain` module holds everything from today, and tomorrow's page includes it. Notice what it doesn't contain: no SQL, no request objects, no JSON parsing. Those belong to the layers you'll build next, and they'll call these rules rather than repeat them.

## Quiz

## Before you go

Read the brief again and find one thing it leaves undecided that today's design decided (for example, whether customers may edit a ticket after an agent has started on it). Tomorrow: the **database**: the schema, migrations that can run on a live system, and the queries the API needs, all in transactions.